# Chapter 4 · Describing One Variable: companion notebook
*Data Science and Artificial Intelligence in Health, Volume 1*

> **This notebook accompanies the book chapter; it does not reproduce it.** Definitions, derivations, and explanations are in the book. Here you run the calculations on the case and check the numbers. All data are **synthetic**, and the values are for teaching purposes only.

**Part A** follows the chapter's case, a cohort of 300 adults admitted with community-acquired pneumonia, through the author's questions; only the sections with a calculation appear (4.2, 4.4 to 4.15, 4.21, 4.22, and 4.23). The book draws the figures; the notebook prints the same numbers. **Part B** gives the written solution of every chapter exercise (1 to 19; statements are in the book) and, for the exercises whose solution needs a calculation (16, 17, and 19), its code and result. The code is Python 3 with pandas. Run the cells in order.

## Part A. The case

**Data.** `data/cohort.csv`: 300 rows, one per patient admitted in 2025, and 11 columns: `patient_id`, `admitted_on`, `sex`, `age_years`, `ward` (medicine, respiratory, geriatrics, intensive care), `severity` (low, moderate, high), `comorbidities`, `sbp_mmhg`, `spo2_pct`, `crp_mg_l`, and `los_days` (length of stay). `data/triage.csv` is a copy of the triage log of Chapter 3, used only by Exercise 17. Each cell carries the number of the book section it belongs to.

### 4.2 A first look at the data with pandas

*How can the cohort file be read and inspected, and what does the library assume about each column?*

In [1]:
# 1. The route of the standard library: a dictionary collects the stays of each ward, and every value is converted from text to a whole number
import csv

stays = {}
with open("data/cohort.csv", newline="") as file:
    for row in csv.DictReader(file):
        if row["ward"] not in stays:
            stays[row["ward"]] = []
        stays[row["ward"]].append(int(row["los_days"]))

# 2. A loop averages each list
for ward in stays:
    days = stays[ward]
    print("Ward", ward + ":", len(days), "patients, mean length of stay", round(sum(days) / len(days), 2), "days")

Ward medicine: 136 patients, mean length of stay 7.46 days
Ward geriatrics: 49 patients, mean length of stay 7.98 days
Ward respiratory: 88 patients, mean length of stay 6.97 days
Ward intensive care: 27 patients, mean length of stay 11.37 days


In [2]:
# 3. The route of pandas: the whole table in one object (a DataFrame), one row per patient and one column per variable
import pandas as pd

df = pd.read_csv("data/cohort.csv")
print("The cohort has", df.shape[0], "rows (one per patient) and", df.shape[1], "columns.")
print("Storage type of los_days:", df.dtypes["los_days"], "; of severity:", df.dtypes["severity"])

# 4. The first three records
print(df.head(3))

The cohort has 300 rows (one per patient) and 11 columns.
Storage type of los_days: int64 ; of severity: str
  patient_id admitted_on sex  age_years         ward  severity  comorbidities  \
0       C001  2025-02-18   F         69     medicine       low              0   
1       C002  2025-03-28   M         73   geriatrics  moderate              0   
2       C003  2025-12-28   M         86  respiratory       low              3   

   sbp_mmhg  spo2_pct  crp_mg_l  los_days  
0       134        96       116         1  
1        93        93        69         8  
2        97        95        51         4  


### 4.4 Frequency distribution

*How often does each value occur?*

In [3]:
# 1. Frequency distribution of severity, in the order of the scale: absolute (F), relative in % (Fr), cumulative (Fac), and cumulative relative in % (Frac)
order = ["low", "moderate", "high"]
table = pd.DataFrame({"F": df["severity"].value_counts().reindex(order)})
table["Fr"] = (100 * table["F"] / len(df)).round(2)
table["Fac"] = table["F"].cumsum()
table["Frac"] = (100 * table["Fac"] / len(df)).round(2)
print(table)

# 2. Ages grouped into ten-year classes, closed on the left: 30-39, 40-49, and so on
limits = [30, 40, 50, 60, 70, 80, 90, 100]
age_class = pd.cut(df["age_years"], bins=limits, right=False)
print("Patients in each age class:", age_class.value_counts().sort_index().tolist())

            F     Fr  Fac    Frac
severity                         
low        97  32.33   97   32.33
moderate  129  43.00  226   75.33
high       74  24.67  300  100.00
Patients in each age class: [8, 16, 53, 77, 79, 50, 17]


### 4.5 Drawing with matplotlib

*How is a variable drawn in Python?*

In [4]:
# 1. The heights of the bars of the severity chart are the counts of each class, in the order of the scale (the book draws them)
counts = df["severity"].value_counts().reindex(order)
print("Bar heights:", counts.tolist())

Bar heights: [97, 129, 74]


### 4.6 Bar chart

*How should the categories be shown in a bar chart?*

In [5]:
# 1. The bars of the wards, from the most to the least frequent (the book draws them)
wards_count = df["ward"].value_counts()
print("Patients by ward, from the most to the least frequent:", wards_count.tolist())

Patients by ward, from the most to the least frequent: [136, 88, 49, 27]


### 4.7 Pareto chart

*Which reasons account for most of the delays?*

In [6]:
# 1. Reasons for delay, from the most to the least frequent, and their cumulative percentages (the book draws the chart)
reasons = pd.Series({"radiograph": 31, "not suspected": 19,
                     "late assessment": 14, "drug unavailable": 9,
                     "venous access": 6, "other": 3})
cumulative = 100 * reasons.cumsum() / reasons.sum()
print("Cumulative percentages:", cumulative.round(1).tolist())

Cumulative percentages: [37.8, 61.0, 78.0, 89.0, 96.3, 100.0]


### 4.8 Histogram

*What shape does length of stay take?*

In [7]:
# 1. Counts of length of stay in classes of five days: 1-5, 6-10, and so on up to 66 (the book draws the histogram)
import numpy as np

counts, limits = np.histogram(df["los_days"], bins=range(1, 67, 5))
print("Counts of the first six classes:", counts[:6].tolist())

Counts of the first six classes: [148, 93, 32, 10, 5, 5]


### 4.9 Stem-and-leaf display

*Can the values themselves be shown while keeping the shape?*

In [8]:
# 1. Stem-and-leaf display of the first 30 systolic pressures: the stem is the tens, the leaf is the units
values = sorted(df["sbp_mmhg"].head(30))
stems = {}
for value in values:
    stem = value // 10
    if stem not in stems:
        stems[stem] = []
    stems[stem].append(value % 10)

# 2. One line per stem
for stem in sorted(stems):
    leaves = ""
    for leaf in stems[stem]:
        leaves = leaves + str(leaf) + " "
    print(str(stem).rjust(3), "|", leaves)

  9 | 3 7 
 10 | 5 
 11 | 1 7 
 12 | 0 3 3 4 5 6 6 7 8 9 
 13 | 3 3 4 6 7 9 
 14 | 0 0 4 6 7 
 15 | 0 5 9 
 16 | 1 


### 4.10 Arithmetic mean

*What is the center of the stays, measured by the arithmetic mean?*

In [9]:
# 1. Mean length of stay: the first ten patients (hand calculation) and the 300 patients
print("Mean of the first ten stays:", df["los_days"].head(10).mean(), "days; of the 300 stays:", round(df["los_days"].mean(), 2), "days")

# 2. By ward: the simple mean of the ward means against the weighted mean (the mean of the cohort)
wards = df.groupby("ward")["los_days"].agg(["count", "mean"])
weighted = (wards["count"] * wards["mean"]).sum() / wards["count"].sum()
print("Simple mean of the ward means:", round(wards["mean"].mean(), 2), "days; weighted mean:", round(weighted, 2), "days")

Mean of the first ten stays: 5.8 days; of the 300 stays: 7.75 days
Simple mean of the ward means: 8.44 days; weighted mean: 7.75 days


### 4.11 Median

*What is the median stay, and does it resist unusual values?*

In [10]:
# 1. Median length of stay: the first ten patients (hand calculation) and the 300 patients
print("Median of the first ten stays:", df["los_days"].head(10).median(), "days; of the 300 stays:", df["los_days"].median(), "days")

# 2. A typing error in a copy of the column: 620 days typed instead of 62
typo = df["los_days"].replace(62, 620)
print("With the typing error: mean", round(typo.mean(), 2), "days; median", typo.median(), "days")

# 3. Patients whose stays are shorter than the mean
print("Patients with stays shorter than the mean:", (df["los_days"] < df["los_days"].mean()).sum())

Median of the first ten stays: 5.5 days; of the 300 stays: 6.0 days
With the typing error: mean 9.61 days; median 6.0 days
Patients with stays shorter than the mean: 196


### 4.12 Line chart

*How did admissions vary over the twelve months?*

In [11]:
# 1. Admissions in each month of 2025 and their median (the book draws the line chart)
month = pd.to_datetime(df["admitted_on"]).dt.month
monthly = month.value_counts().sort_index()
print("Admissions by month, January to December:", monthly.tolist())
print("Median of the twelve monthly counts:", monthly.median())

Admissions by month, January to December: [39, 40, 32, 16, 7, 14, 8, 19, 13, 29, 33, 50]
Median of the twelve monthly counts: 24.0


### 4.13 Mode

*Which value occurs most often?*

In [12]:
# 1. The mode of the ward and of the stay (a method that returns every value tied for the highest frequency)
print("Modal ward:", df["ward"].mode().tolist(), "; modal length of stay:", df["los_days"].mode().tolist(), "days")

# 2. The mode of a finely measured variable against its median
print("Modal systolic pressure:", df["sbp_mmhg"].mode().tolist(), "mmHg; median:", df["sbp_mmhg"].median(), "mmHg")

Modal ward: ['medicine'] ; modal length of stay: [3] days
Modal systolic pressure: [133] mmHg; median: 126.0 mmHg


### 4.14 Quantiles: quartiles, deciles, and percentiles

*Which values split the stays into quarters and tenths?*

In [13]:
# 1. Quartiles and the 90th percentile of the first ten stays (hand calculation)
print("First ten stays - quartile 1, quartile 3, and 90th percentile:", df["los_days"].head(10).quantile([0.25, 0.75, 0.90]).tolist())

# 2. The 10th, 25th, 50th, 75th, and 90th percentiles of the 300 stays
print("300 stays - percentiles 10, 25, 50, 75, and 90:", df["los_days"].quantile([0.10, 0.25, 0.50, 0.75, 0.90]).tolist())

First ten stays - quartile 1, quartile 3, and 90th percentile: [4.0, 7.75, 11.0]
300 stays - percentiles 10, 25, 50, 75, and 90: [2.0, 3.0, 6.0, 9.25, 14.0]


### 4.15 Boxplot and outliers

*Which stays are outliers?*

In [14]:
# 1. Quartiles, interquartile range, and the two fences (1.5 and 3 times the interquartile range above the third quartile)
los = df["los_days"]
q1 = los.quantile(0.25)
q3 = los.quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
extreme = q3 + 3 * iqr
print("Interquartile range:", iqr, "days; lower fence:", q1 - 1.5 * iqr, "days; upper fence:", upper, "days; extreme fence:", extreme, "days")

# 2. The upper whisker and the flagged stays
print("Longest stay inside the fence:", los[los <= upper].max(), "days")
print("Stays beyond the upper fence:", (los > upper).sum(), "; beyond the extreme fence:", (los > extreme).sum())

Interquartile range: 6.25 days; lower fence: -6.375 days; upper fence: 18.625 days; extreme fence: 28.0 days
Longest stay inside the fence: 18 days
Stays beyond the upper fence: 23 ; beyond the extreme fence: 8


### 4.21 Skewness

*Is the distribution symmetric?*

In [15]:
# 1. Skewness (adjusted for samples) of three variables: positive for a longer right tail, negative for a longer left tail
for name in ["los_days", "spo2_pct", "sbp_mmhg"]:
    print("Skewness of", name + ":", round(df[name].skew(), 2))

Skewness of los_days: 3.16
Skewness of spo2_pct: -1.42
Skewness of sbp_mmhg: -0.06


### 4.22 Kurtosis

*How heavy are the tails?*

In [16]:
# 1. Excess kurtosis (adjusted for samples) of the same three variables: zero for the normal distribution
for name in ["los_days", "spo2_pct", "sbp_mmhg"]:
    print("Excess kurtosis of", name + ":", round(df[name].kurt(), 2))

Excess kurtosis of los_days: 14.23
Excess kurtosis of spo2_pct: 2.54
Excess kurtosis of sbp_mmhg: -0.24


### 4.23 Table 1: describing the sample

*How is the whole cohort described in the first table of the manuscript?*

In [17]:
# 1. Non-missing values and summary of the numeric columns of Table 1
cols = ["age_years", "sex", "severity", "comorbidities", "sbp_mmhg", "spo2_pct", "crp_mg_l"]
print("Non-missing values of each column:", df[cols].count().tolist())
print(df[cols].describe().round(1))

# 2. Counts and percentages of the categories
sex_counts = df["sex"].value_counts().reindex(["F", "M"])
print("Sex (F, M): counts", sex_counts.tolist(), "; percentages", (100 * sex_counts / len(df)).round(1).tolist())
severity_counts = df["severity"].value_counts().reindex(["low", "moderate", "high"])
print("Severity (low, moderate, high): counts", severity_counts.tolist(), "; percentages", (100 * severity_counts / len(df)).round(1).tolist())

Non-missing values of each column: [300, 300, 300, 300, 300, 300, 300]
       age_years  comorbidities  sbp_mmhg  spo2_pct  crp_mg_l
count      300.0          300.0     300.0     300.0     300.0
mean        68.5            2.0     126.1      95.2     108.7
std         13.8            1.4      18.8       3.5     112.0
min         31.0            0.0      73.0      80.0       8.0
25%         59.0            1.0     114.0      94.0      42.0
50%         69.0            2.0     126.0      96.0      73.5
75%         78.0            3.0     139.0      98.0     132.8
max         97.0            6.0     172.0     100.0     982.0
Sex (F, M): counts [134, 166] ; percentages [44.7, 55.3]
Severity (low, moderate, high): counts [97, 129, 74] ; percentages [32.3, 43.0, 24.7]


## Part B. Exercise solutions

### 4.25 Exercise 1. Discharge destination

| Destination | $F_i$ | $Fr_i$ (%) |
|---|---|---|
| Home | 52 | 65.00 |
| Home with nursing support | 14 | 17.50 |
| Rehabilitation unit | 9 | 11.25 |
| Long-term care facility | 5 | 6.25 |
| Total | 80 | 100.00 |


The variable is nominal: the destinations have no order that the data impose.
Absolute and relative frequencies are meaningful. The cumulative columns are not,
because they would change with the arbitrary order of the rows.

### 4.25 Exercise 2. Emergency visits

(a) The absolute frequencies 12, 11, 8, 5, 3, 1 give relative frequencies of 30.0, 27.5,
20.0, 12.5, 7.5, and 2.5%, cumulative frequencies of 12, 23, 31, 36, 39, 40, and
cumulative percentages of 30.0, 57.5, 77.5, 90.0, 97.5, 100. Two or more visits:
$8+5+3+1=17$ patients, $17/40=42.5\%$, which is also $100-57.5$. (b) The mean is
$(0\times12+1\times11+2\times8+3\times5+4\times3+6\times1)/40=60/40=1.5$ visits.

### 4.25 Exercise 3. Door-to-needle times

Sturges' expression gives $k=1+3.322\times\log_{10}30\approx5.9$, about six
classes; the range is $118-18=100$ minutes, so $h\approx17$. Classes 20 minutes
wide from 0 are easier to read and give six classes:

| Minutes | $F_i$ | $Fr_i$ (%) | $Fac_i$ | $Frac_i$ (%) |
|---|---|---|---|---|
| $[0,20)$ | 1 | 3.33 | 1 | 3.33 |
| $[20,40)$ | 12 | 40.00 | 13 | 43.33 |
| $[40,60)$ | 10 | 33.33 | 23 | 76.67 |
| $[60,80)$ | 4 | 13.33 | 27 | 90.00 |
| $[80,100)$ | 2 | 6.67 | 29 | 96.67 |
| $[100,120)$ | 1 | 3.33 | 30 | 100.00 |


Under 60 minutes: 23 of 30 patients, 76.67%. Other round widths are acceptable
if the limits and the rule for boundaries are stated.

### 4.25 Exercise 4. Choosing a graph

(a) Blood group: nominal, bar chart sorted by frequency. Serum sodium: continuous,
histogram. Causes of medication
incidents: nominal, with the aim of setting priorities, Pareto chart. Functional
class: ordinal, bar chart in the order I to IV.
(b) Falls per month over three years: a quantity observed in time order,
line chart with the median as reference.

### 4.25 Exercise 5. Cancelled operations

In decreasing order: no bed 46 (38.3%), patient unfit 28 (cumulative 61.7%),
list overran 22 (80.0%), patient did not attend 12 (90.0%), equipment 9
(97.5%), other 3 (100%). The first three causes account for 96 of 120
cancellations, 80%. Bed availability alone accounts for more than a third and
deserves investigation. Choosing an intervention also requires the harm caused,
preventability, feasibility, and resources required; frequency alone does not
establish the best first action.

### 4.25 Exercise 6. Monthly falls

Sorting the 24 counts destroys their order in time, which is the information a
quality team needs: whether falls rose, fell, or changed after an intervention.
The appropriate display is a line chart of the counts in calendar order, a run
chart, with the median of the series as a horizontal reference line.

### 4.25 Exercise 7. Potassium

The sum is 39.9, so the mean is $39.9/9\approx4.43$ mmol/L. In order: 3.8, 3.9,
4.0, 4.0, 4.1, 4.2, 4.4, 4.6, 6.9; the median is the fifth value, 4.1, and the
mode is 4.0. Without the hemolyzed result the mean is $33.0/8\approx4.12$ and the
median is $(4.0+4.1)/2=4.05$. Excluding the laboratory-rejected result moved the mean by 0.31 mmol/L and
the median by 0.05. The exclusion follows the laboratory assessment, not the value
alone; hemolysis by itself does not determine the patient's true potassium.
Record the reason and obtain a valid measurement rather than replace it by the mean.

### 4.25 Exercise 8. A network of hospitals

The weighted mean is
$(1{,}800\times4.2+600\times6.5+200\times9.8)/2{,}600=13{,}420/2{,}600\approx5.16$
days. The simple mean, $(4.2+6.5+9.8)/3\approx6.83$ days, gives hospital C, with
200 discharges, the same influence as hospital A, with 1,800. Each product
$n_j\bar{x}_j$ is a hospital's bed-days, and the network's average is total
bed-days over total discharges.

### 4.25 Exercise 9. Occupancy and bed-days per exit

Available bed-days are $500\times31=15{,}500$. Occupancy is
$12{,}400/15{,}500=0.80$, or 80%. Bed-days per exit are $12{,}400/1{,}550=8.0$
days per exit in that month. This period ratio is not necessarily the mean of
complete stays of the 1,550 patients who left. A patient admitted before the
month contributes only this month's bed-days to the numerator, while one still
hospitalized at month end contributes bed-days without an exit. To use
$n\bar{x}=\sum x$, the numerator must be the complete stays of exactly the
patients counted in the denominator. Occupancy and the period ratio remain
validly calculated; their operational definitions answer different questions.

### 4.25 Exercise 10. Quartiles and fences

With $n=30$: the median has position $29\times0.5+1=15.5$, between 41 and 42, so
$Md=41.5$ minutes. $Q_1$: $h=8.25$, between 33 and 34, $Q_1=33.25$. $Q_3$:
$h=22.75$, between 55 and 58, $Q_3=55+0.75\times3=57.25$. $IQR=24$. Fences:
$33.25-36=-2.75$ and $57.25+36=93.25$; extreme fence $57.25+72=129.25$. The
candidates are 96 and 118 minutes, both moderate. The upper whisker ends at 83.
The two values are checked against the records. If they are correct they stay:
they are the patients the service most needs to learn from.

### 4.25 Exercise 11. A target as a percentile

The target says that the 90th percentile of door-to-needle time should be at most
60 minutes. $P_{90}$ has position $29\times0.9+1=27.1$, between 75 and 83:
$P_{90}=75+0.1\times8=75.8$ minutes. The target was not met: the direct check gives
$23/30=76.67\%$ treated within 60 minutes. For operational targets, use that
count as the decision rule. With small samples, the interpolated $P_{90}$ need
not give exactly the same boundary decision as requiring at least 90% of
observations below a cutoff.

### 4.25 Exercise 12. Variance by hand

The mean is $780/6=130$ mmHg. The deviations are $-12,-6,0,6,12,0$ and their
squares add to $144+36+0+36+144+0=360$. Then
$s^2=360/5=72$ mmHg$^2$ and $s=\sqrt{72}\approx8.49$ mmHg.

### 4.25 Exercise 13. An ambiguous $\pm$

If 0.3 is the standard error, $s=SE\times\sqrt{n}=0.3\times10=3.0$ g/dL.
If it is the standard deviation, the spread is 0.3 g/dL. Both readings are
arithmetically possible; neither the sample size nor the reported mean resolves
the missing label. The authors should identify the statistic explicitly, for
example “mean 12.2 g/dL (SD 3.0)” if 0.3 was the SE, or “mean 12.2 g/dL
(SD 0.3)” if it was the SD. No claim about a percentage within those limits is
justified without a distributional assumption. A 95% confidence interval
would express uncertainty about the population mean; the SD describes
variation among the observed patients. Reporting one does not replace the other.

### 4.25 Exercise 14. Relative precision

$CV_1=100\times0.12/2.40=5.0\%$ and $CV_2=100\times0.30/7.50=4.0\%$. Method 2 has
the larger standard deviation and is the more precise in relative terms. Degrees
Celsius form an interval scale: converting to Fahrenheit changes the mean by
multiplying and adding, and the standard deviation by multiplying only, so the
coefficient changes with the unit and describes no property of the patients.

### 4.25 Exercise 15. Skewness from hypothetical summaries

The mean, 42 days, is 7 days above the median, 35. The third quartile is 20 days above
the median and the first quartile 13 days below it. Both comparisons suggest right
asymmetry. The authors should lead with the median and the quartiles, 35 days [22, 55]. The
mean remains relevant for bed-days. Without the data, the summaries only suggest the shape.

### 4.25 Exercise 16. C-reactive protein

The histogram has a tall peak below 100 mg/L and a long right tail reaching 982.
Mean 108.7, standard deviation 112.0, median 73.5, quartiles 42.0 and 132.75,
adjusted skewness 3.34; 20 values lie above the upper fence of 268.9 mg/L. The
large standard deviation is consistent with the extreme upper values; it is
not an error merely because it exceeds the mean. A
suitable sentence: “Median C-reactive protein at admission was 73.5 mg/L
(first and third quartiles, 42.0 and 132.8 mg/L).”

In [18]:
# 1. C-reactive protein: summary, adjusted skewness, upper fence, and values beyond it
crp = df["crp_mg_l"]
print(crp.describe().round(1))
print("Adjusted skewness:", round(crp.skew(), 2))
q1 = crp.quantile(0.25)
q3 = crp.quantile(0.75)
upper = q3 + 1.5 * (q3 - q1)
print("Upper fence:", upper, "mg/L; values beyond it:", (crp > upper).sum())

count    300.0
mean     108.7
std      112.0
min        8.0
25%       42.0
50%       73.5
75%      132.8
max      982.0
Name: crp_mg_l, dtype: float64
Adjusted skewness: 3.34
Upper fence: 268.875 mg/L; values beyond it: 20


### 4.25 Exercise 17. Triage waiting times

There are 197 known waiting times. Mean 74.1 minutes, median 67, quartiles 36 and
95, $IQR=59$, upper fence $95+88.5=183.5$ minutes, 5 candidate outliers, and
$P_{90}=147.8$ minutes. The median answers the question about a typical recorded wait:
at least half of the 197 observed waits were 67 minutes or less. The 90th percentile answers the
question a service usually cares about next: one patient in ten waited longer
than about two and a half hours.

In [19]:
# 1. Waiting times of the triage log of Chapter 3 (a copy of the file is in data/triage.csv), in minutes; visits with no time seen are left out
log = pd.read_csv("data/triage.csv", parse_dates=["arrived_at", "seen_at"])
wait = ((log["seen_at"] - log["arrived_at"]).dt.total_seconds() / 60).dropna()
print("Known waiting times:", len(wait), "; mean:", round(wait.mean(), 1), "minutes; median:", wait.median(), "minutes")

# 2. Quartiles, upper fence, candidate outliers, and the 90th percentile
q1 = wait.quantile(0.25)
q3 = wait.quantile(0.75)
upper = q3 + 1.5 * (q3 - q1)
print("Quartiles:", q1, "and", q3, "; upper fence:", upper, "minutes; waits beyond it:", (wait > upper).sum(), "; 90th percentile:", round(wait.quantile(0.9), 1), "minutes")

Known waiting times: 197 ; mean: 74.1 minutes; median: 67.0 minutes
Quartiles: 36.0 and 95.0 ; upper fence: 183.5 minutes; waits beyond it: 5 ; 90th percentile: 147.8 minutes


### 4.25 Exercise 18. Repairing a Table 1

Put the number included in the heading and identify the population and period
in the caption. Replace mean severity by counts and percentages for low,
moderate, and high in that order. Replace the unlabeled age expression by
mean (SD), calculated from the individual ages; the standard error of 0.8
is not the spread of patients' ages. If these are the chapter's records, age
is 68.5 years (SD 13.8). State the denominator of the male percentage,
166/300 = 55.3%, and report whether values are missing for each variable.
The table characterizes those studied; the eligibility criteria belong in
the study methods.

### 4.25 Exercise 19. Adding ward to Table 1

Medicine: 136 (45.3%); respiratory: 88 (29.3%); geriatrics: 49 (16.3%);
intensive care: 27 (9.0%). The denominator is 300, and there are no missing
ward values. The percentages add to 99.9% because of rounding. Each count
summarizes ward alone; no cross-tabulation or within-ward comparison with
another variable has been calculated.

In [20]:
# 1. Counts and percentages of the wards, and the number of missing ward values
ward_counts = df["ward"].value_counts()
print("Patients by ward:", ward_counts.tolist())
print("Percentages:", (100 * ward_counts / len(df)).round(1).tolist())
print("Missing ward values:", len(df) - df["ward"].count())

Patients by ward: [136, 88, 49, 27]
Percentages: [45.3, 29.3, 16.3, 9.0]
Missing ward values: 0


---
*Synthetic data (`generate_data.py`, seed 4). Teaching values only; nothing here is an estimate for a real hospital.*